# Ultimate Modular Tabular Base Notebook

1. **35% Private Leaderboard**: Menggabungkan 5 keluarga model berbeda (**LightGBM, CatBoost, XGBoost, HistGradientBoosting, Regularized Linear**) dengan **Greedy Hill Climbing**, **Nested L2 Stacking**, dan **Rank Averaging** pada prediksi Out-Of-Fold (OOF).
2. **30% Presentasi**: Mengekspor otomatis **6 grafik diagnostik & bisnis beresolusi 300 DPI** ke dalam `presentation_assets_ultimate.zip`.
3. **25% Ketepatan Metode**: Memisahkan **Stateless Hook** (yang Anda kustomisasi sendiri) dan **Stateful In-Fold Pipeline** (`TargetEncoder`, `GroupAgg`, `PCA`, `KMeans` dihitung *strictly* di dalam fold `X_train` saja) sehingga **0% Data Leakage**.
4. **10% Babak Penyisihan**: Dilengkapi *auto memory downcasting*, deteksi tipe kolom otomatis, serta *assertion checks* ketat pada 3 file *submission* akhir.

---
* **Langkah 1**: Sesuaikan nama file dan target pada **Sel 2 (`CONFIG`)**.
* **Langkah 2**: Masukkan logika *Data Preprocessing* dan *Feature Engineering* Anda pada **Sel 4 (`user_custom_preprocessing` & `user_custom_feature_engineering`)**.
* **Langkah 3**: Jalankan seluruh sel lainnya tanpa perlu mengubah mesin CV, pelatihan model, ensemble, maupun ekspor grafik presentasi.


## 1. Import Pustaka & Urutan Inisialisasi OpenMP
> **Catatan Teknis Penting:** Pustaka GBDT (`lightgbm`, `xgboost`, `catboost`) wajib di-import pada baris paling atas sebelum `numpy`, `scipy`, dan `sklearn` untuk mencegah konflik *runtime* OpenMP DLL (`vcomp140.dll` / `libiomp5md.dll`) pada sistem operasi Windows.


In [ ]:
# PENTING: Import GBDT di baris paling atas sebelum NumPy/SciPy/Scikit-Learn
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

import os
import gc
import time
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import rankdata

from sklearn.model_selection import (
    StratifiedKFold, KFold, GroupKFold, TimeSeriesSplit, cross_val_predict
)
from sklearn.metrics import (
    roc_auc_score, f1_score, log_loss, accuracy_score,
    mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

try:
    import shap
except ImportError:
    shap = None

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 150


## 2. Konfigurasi Pusat (`CONFIG`) & Mesin Evaluasi Metrik
Ubah nilai di dalam `CONFIG` sesuai spesifikasi soal lomba:
* `TASK`: `'binary'`, `'multiclass'`, atau `'regression'`.
* `METRIC`: `'roc_auc'`, `'f1'`, `'f1_macro'`, `'accuracy'`, `'log_loss'`, `'rmse'`, atau `'mae'`.
* `CV_STRATEGY`: `'stratified'` (klasifikasi standar), `'group'` (jika ada ID pengguna/entitas berulang di `GROUP_COL`), `'timeseries'` (data berurutan waktu), atau `'kfold'` (regresi standar).


In [ ]:
CONFIG = {
    "TRAIN_PATH": "train.csv",
    "TEST_PATH": "test.csv",
    "SAMPLE_SUB_PATH": "sample_submission.csv",
    "EXPORT_DIR": "./presentation_assets_ultimate/",
    "ID_COL": "id",
    "TARGET_COL": "target",
    "GROUP_COL": None,               # Isi nama kolom grup jika memakai CV_STRATEGY = 'group'
    "TASK": "binary",                # Pilihan: 'binary', 'multiclass', 'regression'
    "METRIC": "roc_auc",             # Pilihan: 'roc_auc', 'f1', 'f1_macro', 'accuracy', 'log_loss', 'rmse', 'mae'
    "CV_STRATEGY": "stratified",     # Pilihan: 'stratified', 'kfold', 'group', 'timeseries'
    "N_SPLITS": 5,
    "SEED": 42,
    "N_CLUSTERS": 5,
    "N_PCA_COMPONENTS": 3,
    "EARLY_STOPPING_ROUNDS": 80,
    "N_ESTIMATORS": 1000,
    "USE_IN_FOLD_TE": True,          # Out-of-Fold Target Encoding otomatis di dalam CV
    "USE_IN_FOLD_CLUSTER_PCA": True, # Unsupervised K-Means & PCA otomatis di dalam CV
}

os.makedirs(CONFIG["EXPORT_DIR"], exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CONFIG["SEED"])


def reduce_mem_usage(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Kompresi tipe data numerik secara aman untuk menghemat RAM hingga 50%."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type) and not pd.api.types.is_bool_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type).startswith("int"):
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage: {start_mem:.2f} MB -> {end_mem:.2f} MB ({100 * (start_mem - end_mem) / max(start_mem, 1e-6):.1f}% reduction)")
    return df


def is_higher_better(metric_name: str) -> bool:
    return metric_name in ["roc_auc", "f1", "f1_macro", "accuracy"]


def calculate_metric(y_true: np.ndarray, y_pred: np.ndarray, metric_name: str, task: str = "binary", threshold: float = 0.5) -> float:
    if metric_name == "roc_auc":
        return float(roc_auc_score(y_true, y_pred) if task == "binary" else roc_auc_score(y_true, y_pred, multi_class="ovr"))
    elif metric_name == "f1_macro":
        preds_bin = (y_pred >= threshold).astype(int) if task == "binary" else np.argmax(y_pred, axis=1)
        return float(f1_score(y_true, preds_bin, average="macro"))
    elif metric_name == "f1":
        preds_bin = (y_pred >= threshold).astype(int)
        return float(f1_score(y_true, preds_bin, average="binary"))
    elif metric_name == "accuracy":
        preds_bin = (y_pred >= threshold).astype(int) if task == "binary" else np.argmax(y_pred, axis=1)
        return float(accuracy_score(y_true, preds_bin))
    elif metric_name == "log_loss":
        return float(log_loss(y_true, np.clip(y_pred, 1e-15, 1 - 1e-15)))
    elif metric_name == "rmse":
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))
    elif metric_name == "mae":
        return float(mean_absolute_error(y_true, y_pred))
    raise ValueError(f"Unknown metric: {metric_name}")


def optimize_threshold(y_true: np.ndarray, y_prob: np.ndarray, metric_name: str = "f1") -> tuple[float, float]:
    best_thresh, best_score = 0.5, -np.inf
    eval_metric = metric_name if metric_name in ["f1", "f1_macro", "accuracy"] else "f1"
    for thresh in np.linspace(0.10, 0.90, 81):
        score = calculate_metric(y_true, y_prob, eval_metric, task="binary", threshold=thresh)
        if score > best_score:
            best_score, best_thresh = score, float(thresh)
    return best_thresh, best_score


def rank_average(preds_list: list[np.ndarray]) -> np.ndarray:
    ranked_preds = [rankdata(p) / len(p) for p in preds_list]
    return np.mean(ranked_preds, axis=0)


## 3. Pemuatan Data & Audit Eksploratif Otomatis (Plot 01)
Sel ini memuat dataset kompetisi, mengompresi penggunaan memori RAM, dan mengekspor grafik `01_eda_target_and_missing.png` (proporsi kelas target serta persentase nilai kosong per kolom).


In [1]:
if not os.path.exists(CONFIG["TRAIN_PATH"]):
    print("[INFO] File CSV kompetisi belum ada. Membuat dataset sintetis realistis (Train & Test independen)...")
    from sklearn.datasets import make_classification
    X_syn, y_syn = make_classification(
        n_samples=3200, n_features=14, n_informative=9, n_redundant=3,
        weights=[0.65, 0.35], flip_y=0.03, random_state=CONFIG["SEED"]
    )
    df_all = pd.DataFrame(X_syn, columns=[f"num_feat_{i}" for i in range(14)])
    rng = np.random.default_rng(CONFIG["SEED"])
    df_all["cat_region"] = rng.choice(["Jakarta", "Surabaya", "Bandung", "Medan", "Semarang"], size=len(df_all), p=[0.4, 0.25, 0.15, 0.1, 0.1])
    df_all["cat_segment"] = rng.choice(["Enterprise", "SME", "Consumer", "Micro"], size=len(df_all))
    df_all["cat_tier"] = rng.choice(["Gold", "Silver", "Bronze"], size=len(df_all), p=[0.2, 0.5, 0.3])

    for col_idx in [1, 4, 7]:
        mask = rng.random(len(df_all)) < 0.06
        df_all.loc[mask, f"num_feat_{col_idx}"] = np.nan

    df_train = df_all.iloc[:2500].copy().reset_index(drop=True)
    df_train[CONFIG["ID_COL"]] = np.arange(1, len(df_train) + 1)
    df_train[CONFIG["TARGET_COL"]] = y_syn[:2500]

    df_test = df_all.iloc[2500:].copy().reset_index(drop=True)
    df_test[CONFIG["ID_COL"]] = np.arange(10001, 10001 + len(df_test))
else:
    df_train = pd.read_csv(CONFIG["TRAIN_PATH"])
    df_test = pd.read_csv(CONFIG["TEST_PATH"])

df_train = reduce_mem_usage(df_train)
df_test = reduce_mem_usage(df_test)
print(f"Dataset Shapes -> Train: {df_train.shape} | Test: {df_test.shape}")

# Plot 01: EDA Target Distribution & Missing Value Profile
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
if CONFIG["TASK"] in ["binary", "multiclass"]:
    val_cnts = df_train[CONFIG["TARGET_COL"]].value_counts().sort_index()
    sns.barplot(x=val_cnts.index.astype(str), y=val_cnts.values, ax=axes[0], palette="Blues_d")
    axes[0].set_title("Target Class Distribution (Train)", fontsize=12, weight="bold")
    axes[0].set_xlabel("Target Class")
    axes[0].set_ylabel("Count")
    for idx, v in enumerate(val_cnts.values):
        axes[0].text(idx, v + max(val_cnts.values) * 0.02, f"{v} ({v/len(df_train)*100:.1f}%)", ha="center", fontweight="bold")
else:
    sns.histplot(df_train[CONFIG["TARGET_COL"]], kde=True, ax=axes[0], color="#2980b9")
    axes[0].set_title("Target Value Distribution (Train)", fontsize=12, weight="bold")

null_pct = (df_train.isnull().sum() / len(df_train) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
if len(null_pct) > 0:
    sns.barplot(x=null_pct.values, y=null_pct.index, ax=axes[1], palette="Reds_r")
    axes[1].set_title("Columns with Missing Values (%)", fontsize=12, weight="bold")
    axes[1].set_xlabel("Missing Percentage (%)")
else:
    axes[1].text(0.5, 0.5, "Zero Missing Values Detected", ha="center", va="center", fontsize=12, weight="bold")
    axes[1].set_title("Missing Value Audit", fontsize=12, weight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["EXPORT_DIR"], "01_eda_target_and_missing.png"), dpi=300)
plt.show()


[INFO] File CSV kompetisi belum ada. Membuat dataset sintetis realistis (Train & Test independen)...
Memory usage: 0.36 MB -> 0.20 MB (45.4% reduction)
Memory usage: 0.10 MB -> 0.05 MB (43.0% reduction)
Dataset Shapes -> Train: (2500, 19) | Test: (700, 18)


## 4. 🛠️ ZONA KUSTOMISASI ANDA: Data Preprocessing & Feature Engineering
> **Bagian ini dirancang khusus agar mudah Anda ubah saat lomba.**
>
> **Aturan Emas Bebas Kebocoran Data (*Zero Data Leakage*):**
> 1. Pada fungsi `user_custom_preprocessing(df)` dan `user_custom_feature_engineering(df)`, lakukan **hanya operasi baris-per-baris (*stateless row-wise*)**:
>    * Konversi tanggal (`pd.to_datetime`), ekstraksi hari/bulan/jam.
>    * Pembersihan string/teks, penggantian kode *missing value* terselubung (`-999`, `"?"`, `"Unknown"` menjadi `np.nan`).
>    * Pembuatan rasio domain (contoh: `pinjaman / pendapatan`), selisih waktu, polinomial, atau transformasi `np.log1p()`.
> 2. **Jangan** menghitung statistik lintas-baris seperti `.mean()`, `.std()`, *Target Encoding*, *Frequency Encoding*, atau *Scaler* di fungsi ini! Seluruh transformasi lintas-baris sudah diotomatisasi secara aman di dalam tiap *fold* CV oleh kelas `InFoldFeaturePipeline` pada **Sel 6**.


In [1]:
def user_custom_preprocessing(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    """
    [HOOK 1 - STATELESS DATA PREPROCESSING]
    Silakan sesuaikan pembersihan data spesifik dataset lomba di bawah ini.
    """
    df_out = df.copy()

    # -------------------------------------------------------------------------
    # CONTOH TEMPLATE PREPROCESSING (Uncomment & sesuaikan nama kolom saat lomba)
    # -------------------------------------------------------------------------
    # 1. Ganti nilai kode anomali menjadi NaN
    # df_out.replace([-999, -9999, "?", "None", "null"], np.nan, inplace=True)

    # 2. Parsing kolom Tanggal (Datetime)
    # if "tanggal_registrasi" in df_out.columns:
    #     dt = pd.to_datetime(df_out["tanggal_registrasi"], errors="coerce")
    #     df_out["reg_year"] = dt.dt.year.astype(np.float32)
    #     df_out["reg_month"] = dt.dt.month.astype(np.float32)
    #     df_out["reg_dayofweek"] = dt.dt.dayofweek.astype(np.float32)
    #     df_out["reg_is_weekend"] = (dt.dt.dayofweek >= 5).astype(np.int8)
    #     df_out.drop(columns=["tanggal_registrasi"], inplace=True)

    # 3. Pembersihan kolom nominal berformat string ("Rp 1.500.000" -> 1500000.0)
    # if "harga_str" in df_out.columns:
    #     df_out["harga_num"] = df_out["harga_str"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
    #     df_out["harga_num"] = pd.to_numeric(df_out["harga_num"], errors="coerce")
    #     df_out.drop(columns=["harga_str"], inplace=True)

    return df_out


def user_custom_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    [HOOK 2 - STATELESS DOMAIN FEATURE ENGINEERING]
    Silakan tambahkan fitur domain spesifik Anda di bawah ini.
    """
    df_out = df.copy()

    exclude_cols = [CONFIG["ID_COL"], CONFIG["TARGET_COL"], CONFIG["GROUP_COL"]]
    raw_cols = [c for c in df_out.columns if c not in exclude_cols]
    num_cols = [c for c in raw_cols if pd.api.types.is_numeric_dtype(df_out[c])]
    cat_cols = [c for c in raw_cols if c not in num_cols]

    # -------------------------------------------------------------------------
    # A. FITUR GENERIK OTOMATIS (Statistik Baris & Interaksi Kategori)
    # -------------------------------------------------------------------------
    if len(num_cols) >= 2:
        df_out["fe_row_null_cnt"] = df_out[num_cols].isnull().sum(axis=1).astype(np.float32)
        df_out["fe_row_mean"] = df_out[num_cols].mean(axis=1).astype(np.float32)
        df_out["fe_row_std"] = df_out[num_cols].std(axis=1).fillna(0).astype(np.float32)
        df_out["fe_row_max"] = df_out[num_cols].max(axis=1).astype(np.float32)
        df_out["fe_row_min"] = df_out[num_cols].min(axis=1).astype(np.float32)
        df_out["fe_row_range"] = (df_out["fe_row_max"] - df_out["fe_row_min"]).astype(np.float32)

    if len(cat_cols) >= 2:
        df_out["fe_cat_cross"] = df_out[cat_cols[0]].astype(str) + "__" + df_out[cat_cols[1]].astype(str)

    # -------------------------------------------------------------------------
    # B. TAMBAHKAN FITUR DOMAIN SPESIFIK LOMBA ANDA DI SINI:
    # -------------------------------------------------------------------------
    # Contoh 1: Rasio finansial / interaksi perkalian & pembagian
    # if "total_tagihan" in df_out.columns and "pendapatan_bulanan" in df_out.columns:
    #     df_out["rasio_tagihan_pendapatan"] = df_out["total_tagihan"] / (df_out["pendapatan_bulanan"] + 1e-5)

    # Contoh 2: Transformasi Logaritmik untuk fitur miring (skewed positif)
    # for skewed_col in ["pendapatan_bulanan", "nominal_transaksi"]:
    #     if skewed_col in df_out.columns:
    #         df_out[f"{skewed_col}_log1p"] = np.log1p(np.maximum(0, df_out[skewed_col]))

    # Contoh 3: Panjang teks untuk kolom deskripsi singkat
    # if "catatan_keluhan" in df_out.columns:
    #     df_out["keluhan_char_len"] = df_out["catatan_keluhan"].fillna("").astype(str).str.len()
    #     df_out["keluhan_word_cnt"] = df_out["catatan_keluhan"].fillna("").astype(str).str.split().str.len()

    return df_out


# Eksekusi Hook Preprocessing & Feature Engineering
df_train_clean = user_custom_feature_engineering(user_custom_preprocessing(df_train, is_train=True))
df_test_clean = user_custom_feature_engineering(user_custom_preprocessing(df_test, is_train=False))

exclude_cols = [CONFIG["ID_COL"], CONFIG["TARGET_COL"], CONFIG["GROUP_COL"]]
base_features = [c for c in df_train_clean.columns if c not in exclude_cols]
cat_features = [c for c in base_features if not pd.api.types.is_numeric_dtype(df_train_clean[c])]
num_features = [c for c in base_features if c not in cat_features]

print(f"Post-Hook Features: {len(base_features)} total ({len(num_features)} numerical, {len(cat_features)} categorical)")


Post-Hook Features: 24 total (20 numerical, 4 categorical)


## 5. Adversarial Validation & Diagnostik Pergeseran Fitur (Plot 02)
Sebelum melatih model utama, kita menguji hipotesis kesamaan distribusi $P(X_{\text{train}}) = P(X_{\text{test}})$:
* Jika **AV ROC-AUC $\approx 0.50$**, distribusi data *train* dan *test* konsisten.
* Jika **AV ROC-AUC $> 0.70$**, lihat grafik kanan pada `02_adversarial_validation_and_drift.png` (*Top 10 Features Driving Train-Test Split*). Fitur yang memiliki skor *adversarial split* sangat tinggi menandakan *feature drift* (contoh: kolom ID berurutan atau *timestamp* yang bocor) dan sebaiknya Anda *drop* di fungsi `user_custom_preprocessing`.


In [1]:
av_tr = df_train_clean[base_features].copy()
av_te = df_test_clean[base_features].copy()
av_tr["is_test"] = 0
av_te["is_test"] = 1
av_df = pd.concat([av_tr, av_te], axis=0).reset_index(drop=True)

if len(cat_features) > 0:
    av_ord = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    av_df[cat_features] = av_ord.fit_transform(av_df[cat_features].astype(str)).astype(np.float32)

av_X = av_df[base_features].astype(np.float32)
av_y = av_df["is_test"].astype(np.int32)
av_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG["SEED"])
av_oof = np.zeros(len(av_df))
av_importances = np.zeros(len(base_features))

for tr_idx, val_idx in av_kf.split(av_X, av_y):
    m_av = lgb.LGBMClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=4, num_leaves=15,
        random_state=CONFIG["SEED"], verbosity=-1, n_jobs=-1
    )
    m_av.fit(av_X.iloc[tr_idx], av_y.iloc[tr_idx])
    av_oof[val_idx] = m_av.predict_proba(av_X.iloc[val_idx])[:, 1]
    av_importances += m_av.feature_importances_ / 5.0

av_auc = roc_auc_score(av_y, av_oof)
print(f"Adversarial Validation ROC-AUC: {av_auc:.4f}")
if av_auc > 0.70:
    print("[WARNING] Terdeteksi pergeseran distribusi (Covariate Shift > 0.70)! Periksa fitur teratas di plot 02.")
else:
    print("[OK] Distribusi Train dan Test konsisten (AUC ~ 0.50). Skema Cross-Validation lokal sangat valid.")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
fpr, tpr, _ = roc_curve(av_y, av_oof)
axes[0].plot(fpr, tpr, color="#e74c3c", lw=2.2, label=f"AV ROC-AUC = {av_auc:.4f}")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", label="Ideal Random (0.50)")
axes[0].set_title("Adversarial Validation ROC (Train vs Test)", fontsize=12, weight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

drift_s = pd.Series(av_importances, index=base_features).sort_values(ascending=False).head(10)
sns.barplot(x=drift_s.values, y=drift_s.index, ax=axes[1], palette="magma")
axes[1].set_title("Top 10 Features Driving Train-Test Split", fontsize=12, weight="bold")
axes[1].set_xlabel("Adversarial Split Importance")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["EXPORT_DIR"], "02_adversarial_validation_and_drift.png"), dpi=300)
plt.show()


Adversarial Validation ROC-AUC: 0.4983
[OK] Distribusi Train dan Test konsisten (AUC ~ 0.50). Skema Cross-Validation lokal sangat valid.


## 6. Mesin Rekayasa Fitur Stateful di Dalam Fold (`InFoldFeaturePipeline`)
Untuk mendapatkan nilai sempurna pada komponen **25% Ketepatan Metode**, seluruh transformasi yang mempelajari parameter dari distribusi baris wajib dijalankan **di dalam masing-masing fold Cross-Validation** (`fit_transform` hanya pada `X_train`, lalu `transform` pada `X_val` dan `X_test`):
1. **In-Fold Frequency Encoding**: Menghitung proporsi kemunculan kategori pada `X_train`.
2. **In-Fold Group Aggregations**: Menghitung rata-rata dan standar deviasi fitur numerik per kategori utama pada `X_train`.
3. **Smoothed Out-Of-Fold Target Encoding (`sklearn.preprocessing.TargetEncoder`)**: Menggunakan *internal cross-fitting* dan *empirical Bayes smoothing* (`smooth="auto"`) sehingga kategori langka menyusut (*shrink*) ke rata-rata global tanpa membocorkan target.
4. **In-Fold K-Means Centroid Distances & PCA**: Memberikan koordinat jarak radial dan proyeksi sumbu miring bagi pohon keputusan.


In [ ]:
class InFoldFeaturePipeline:
    """
    Transformasi stateful di dalam fold CV untuk menjamin 0% Data Leakage.
    """
    def __init__(self, num_cols: list[str], cat_cols: list[str], config: dict, seed: int = 42):
        self.num_cols = list(num_cols)
        self.cat_cols = list(cat_cols)
        self.config = config
        self.seed = seed
        self.freq_maps = {}
        self.group_aggs = {}
        self.main_cat = self.cat_cols[0] if len(self.cat_cols) > 0 else None
        self.top_num_for_agg = self.num_cols[:3]

        self.ord_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.te_encoder = None
        self.num_imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        self.pca = None
        self.kmeans = None

    def fit_transform(self, X_tr: pd.DataFrame, y_tr: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
        df = X_tr.copy()

        # 1. Frequency Encoding (fit on X_train only)
        for c in self.cat_cols:
            s_str = df[c].astype(str)
            freq = (s_str.value_counts() / len(df)).to_dict()
            self.freq_maps[c] = freq
            df[f"{c}_freq"] = s_str.map(freq).fillna(0.0).astype(np.float32)

        # 2. Group Aggregations (fit on X_train only)
        if self.main_cat is not None and len(self.top_num_for_agg) > 0:
            for nc in self.top_num_for_agg:
                grp = df.groupby(df[self.main_cat].astype(str))[nc].agg(["mean", "std"])
                self.group_aggs[nc] = {
                    "mean": grp["mean"].to_dict(),
                    "std": grp["std"].fillna(0.0).to_dict(),
                    "global_mean": float(df[nc].mean())
                }
                key_s = df[self.main_cat].astype(str)
                df[f"{self.main_cat}_{nc}_grp_mean"] = key_s.map(self.group_aggs[nc]["mean"]).fillna(self.group_aggs[nc]["global_mean"]).astype(np.float32)
                df[f"{self.main_cat}_{nc}_grp_std"] = key_s.map(self.group_aggs[nc]["std"]).fillna(0.0).astype(np.float32)

        # 3. Smoothed Out-Of-Fold Target Encoding (via Scikit-Learn TargetEncoder)
        if self.config["USE_IN_FOLD_TE"] and len(self.cat_cols) > 0 and self.config["TASK"] in ["binary", "regression"]:
            t_type = "binary" if self.config["TASK"] == "binary" else "continuous"
            self.te_encoder = TargetEncoder(smooth="auto", cv=5, target_type=t_type, random_state=self.seed)
            cat_str_df = df[self.cat_cols].astype(str)
            te_vals = self.te_encoder.fit_transform(cat_str_df, y_tr)
            for idx, c in enumerate(self.cat_cols):
                df[f"{c}_te"] = te_vals[:, idx].astype(np.float32)

        # 4. Numerical Imputation, Scaling, PCA & K-Means Centroid Distances
        num_mat = self.num_imputer.fit_transform(df[self.num_cols])
        scaled_mat = self.scaler.fit_transform(num_mat)

        if self.config["USE_IN_FOLD_CLUSTER_PCA"] and len(self.num_cols) >= self.config["N_PCA_COMPONENTS"]:
            n_pca = min(self.config["N_PCA_COMPONENTS"], len(self.num_cols))
            self.pca = PCA(n_components=n_pca, random_state=self.seed)
            pca_feats = self.pca.fit_transform(scaled_mat)
            for i in range(n_pca):
                df[f"infold_pca_{i}"] = pca_feats[:, i].astype(np.float32)

            self.kmeans = KMeans(n_clusters=self.config["N_CLUSTERS"], random_state=self.seed, n_init=10)
            dist_feats = self.kmeans.fit_transform(scaled_mat)
            for i in range(self.config["N_CLUSTERS"]):
                df[f"infold_dist_clust_{i}"] = dist_feats[:, i].astype(np.float32)

        # 5. Ordinal Encoding for Categoricals (handles unseen categories in test with -1)
        if len(self.cat_cols) > 0:
            df[self.cat_cols] = self.ord_encoder.fit_transform(df[self.cat_cols].astype(str)).astype(np.int32)

        # Prepare clean scaled matrix for linear models
        all_feat_cols = list(df.columns)
        self.final_imputer = SimpleImputer(strategy="median")
        self.final_scaler = StandardScaler()
        df_linear = pd.DataFrame(
            self.final_scaler.fit_transform(self.final_imputer.fit_transform(df)),
            columns=all_feat_cols, index=df.index
        )
        return df, df_linear

    def transform(self, X_new: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        df = X_new.copy()

        for c in self.cat_cols:
            s_str = df[c].astype(str)
            df[f"{c}_freq"] = s_str.map(self.freq_maps[c]).fillna(0.0).astype(np.float32)

        if self.main_cat is not None and len(self.top_num_for_agg) > 0:
            key_s = df[self.main_cat].astype(str)
            for nc in self.top_num_for_agg:
                info = self.group_aggs[nc]
                df[f"{self.main_cat}_{nc}_grp_mean"] = key_s.map(info["mean"]).fillna(info["global_mean"]).astype(np.float32)
                df[f"{self.main_cat}_{nc}_grp_std"] = key_s.map(info["std"]).fillna(0.0).astype(np.float32)

        if self.te_encoder is not None:
            cat_str_df = df[self.cat_cols].astype(str)
            te_vals = self.te_encoder.transform(cat_str_df)
            for idx, c in enumerate(self.cat_cols):
                df[f"{c}_te"] = te_vals[:, idx].astype(np.float32)

        num_mat = self.num_imputer.transform(df[self.num_cols])
        scaled_mat = self.scaler.transform(num_mat)

        if self.pca is not None and self.kmeans is not None:
            pca_feats = self.pca.transform(scaled_mat)
            for i in range(pca_feats.shape[1]):
                df[f"infold_pca_{i}"] = pca_feats[:, i].astype(np.float32)

            dist_feats = self.kmeans.transform(scaled_mat)
            for i in range(dist_feats.shape[1]):
                df[f"infold_dist_clust_{i}"] = dist_feats[:, i].astype(np.float32)

        if len(self.cat_cols) > 0:
            df[self.cat_cols] = self.ord_encoder.transform(df[self.cat_cols].astype(str)).astype(np.int32)

        all_feat_cols = list(df.columns)
        df_linear = pd.DataFrame(
            self.final_scaler.transform(self.final_imputer.transform(df)),
            columns=all_feat_cols, index=df.index
        )
        return df, df_linear


## 7. Pelatihan 5 Keluarga Model Beragam (Level-1 Model Zoo)
Untuk memaksimalkan skor **35% Private Leaderboard**, kita melatih 5 arsitektur dengan struktur pembelahan pohon dan bias induktif yang berbeda di setiap *fold*:
1. **LightGBM (`lgb`)**: Pertumbuhan pohon *leaf-wise* dengan regularisasi L1/L2 dan *feature subsampling*.
2. **CatBoost (`cat`)**: Pohon simetris (*oblivious trees*) yang menggunakan kriteria *split* seragam pada setiap kedalaman pohon, sangat kebal terhadap *overfitting*.
3. **XGBoost (`xgb`)**: Pertumbuhan pohon *depth-wise* dengan kontrol bobot turunan kedua (*Hessian*).
4. **HistGradientBoosting (`hgb`)**: Implementasi GBDT berbasis *histogram binning* asli Scikit-Learn untuk menambah variasi struktur pohon.
5. **L2-Regularized Linear (`linear`)**: Model `LogisticRegression` / `Ridge` pada fitur terstandarisasi untuk menangkap tren global linear.


In [1]:
X_raw = df_train_clean[base_features].reset_index(drop=True)
y_raw = df_train_clean[CONFIG["TARGET_COL"]].reset_index(drop=True)
X_test_raw = df_test_clean[base_features].reset_index(drop=True)

if CONFIG["CV_STRATEGY"] == "stratified" and CONFIG["TASK"] in ["binary", "multiclass"]:
    splitter = StratifiedKFold(n_splits=CONFIG["N_SPLITS"], shuffle=True, random_state=CONFIG["SEED"])
    splits = list(splitter.split(X_raw, y_raw))
elif CONFIG["CV_STRATEGY"] == "group" and CONFIG["GROUP_COL"] is not None:
    splitter = GroupKFold(n_splits=CONFIG["N_SPLITS"])
    splits = list(splitter.split(X_raw, y_raw, groups=df_train_clean[CONFIG["GROUP_COL"]]))
elif CONFIG["CV_STRATEGY"] == "timeseries":
    splitter = TimeSeriesSplit(n_splits=CONFIG["N_SPLITS"])
    splits = list(splitter.split(X_raw))
else:
    splitter = KFold(n_splits=CONFIG["N_SPLITS"], shuffle=True, random_state=CONFIG["SEED"])
    splits = list(splitter.split(X_raw))

model_names = ["lgb", "cat", "xgb", "hgb", "linear"]
oof_preds = {m: np.zeros(len(X_raw), dtype=np.float64) for m in model_names}
test_preds = {m: np.zeros(len(X_test_raw), dtype=np.float64) for m in model_names}
fold_scores = {m: [] for m in model_names}
fitted_lgb_models = []
last_val_X = None

t0 = time.time()
n_folds_actual = len(splits)

for fold, (tr_idx, val_idx) in enumerate(splits):
    X_tr_raw, y_tr = X_raw.iloc[tr_idx], y_raw.iloc[tr_idx]
    X_va_raw, y_va = X_raw.iloc[val_idx], y_raw.iloc[val_idx]

    # Fit InFoldFeaturePipeline HANYA pada X_tr_raw
    fe_pipe = InFoldFeaturePipeline(num_features, cat_features, CONFIG, seed=CONFIG["SEED"] + fold)
    X_tr, X_tr_lin = fe_pipe.fit_transform(X_tr_raw, y_tr)
    X_va, X_va_lin = fe_pipe.transform(X_va_raw)
    X_te, X_te_lin = fe_pipe.transform(X_test_raw)
    last_val_X = X_va

    # 1. LightGBM (Leaf-wise GBDT)
    lgb_params = {
        "objective": "binary" if CONFIG["TASK"] == "binary" else "regression",
        "metric": "auc" if CONFIG["METRIC"] == "roc_auc" else ("logloss" if CONFIG["TASK"] != "regression" else "rmse"),
        "learning_rate": 0.04,
        "num_leaves": 31,
        "subsample_for_bin": 50000,
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "reg_alpha": 1.0,
        "reg_lambda": 2.0,
        "n_estimators": CONFIG["N_ESTIMATORS"],
        "random_state": CONFIG["SEED"] + fold,
        "verbosity": -1,
        "n_jobs": -1,
    }
    m_lgb = lgb.LGBMClassifier(**lgb_params) if CONFIG["TASK"] != "regression" else lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(CONFIG["EARLY_STOPPING_ROUNDS"], verbose=False)])
    p_va_lgb = m_lgb.predict_proba(X_va)[:, 1] if CONFIG["TASK"] == "binary" else m_lgb.predict(X_va)
    p_te_lgb = m_lgb.predict_proba(X_te)[:, 1] if CONFIG["TASK"] == "binary" else m_lgb.predict(X_te)
    oof_preds["lgb"][val_idx] = p_va_lgb
    test_preds["lgb"] += p_te_lgb / n_folds_actual
    fold_scores["lgb"].append(calculate_metric(y_va, p_va_lgb, CONFIG["METRIC"], CONFIG["TASK"]))
    fitted_lgb_models.append(m_lgb)

    # 2. CatBoost (Symmetric Oblivious Trees)
    cb_params = {
        "iterations": CONFIG["N_ESTIMATORS"],
        "learning_rate": 0.045,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "random_seed": CONFIG["SEED"] + fold,
        "verbose": False,
        "early_stopping_rounds": CONFIG["EARLY_STOPPING_ROUNDS"],
    }
    m_cat = cb.CatBoostClassifier(**cb_params) if CONFIG["TASK"] != "regression" else cb.CatBoostRegressor(**cb_params)
    m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
    p_va_cat = m_cat.predict_proba(X_va)[:, 1] if CONFIG["TASK"] == "binary" else m_cat.predict(X_va)
    p_te_cat = m_cat.predict_proba(X_te)[:, 1] if CONFIG["TASK"] == "binary" else m_cat.predict(X_te)
    oof_preds["cat"][val_idx] = p_va_cat
    test_preds["cat"] += p_te_cat / n_folds_actual
    fold_scores["cat"].append(calculate_metric(y_va, p_va_cat, CONFIG["METRIC"], CONFIG["TASK"]))

    # 3. XGBoost (Depth-wise GBDT)
    xgb_params = {
        "n_estimators": CONFIG["N_ESTIMATORS"],
        "learning_rate": 0.04,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.0,
        "reg_lambda": 2.0,
        "random_state": CONFIG["SEED"] + fold,
        "early_stopping_rounds": CONFIG["EARLY_STOPPING_ROUNDS"],
        "n_jobs": -1,
        "eval_metric": "auc" if CONFIG["METRIC"] == "roc_auc" else ("logloss" if CONFIG["TASK"] != "regression" else "rmse"),
    }
    m_xgb = xgb.XGBClassifier(**xgb_params) if CONFIG["TASK"] != "regression" else xgb.XGBRegressor(**xgb_params)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    p_va_xgb = m_xgb.predict_proba(X_va)[:, 1] if CONFIG["TASK"] == "binary" else m_xgb.predict(X_va)
    p_te_xgb = m_xgb.predict_proba(X_te)[:, 1] if CONFIG["TASK"] == "binary" else m_xgb.predict(X_te)
    oof_preds["xgb"][val_idx] = p_va_xgb
    test_preds["xgb"] += p_te_xgb / n_folds_actual
    fold_scores["xgb"].append(calculate_metric(y_va, p_va_xgb, CONFIG["METRIC"], CONFIG["TASK"]))

    # 4. Scikit-Learn HistGradientBoosting
    hgb_params = {
        "max_iter": 400,
        "learning_rate": 0.04,
        "max_leaf_nodes": 31,
        "l2_regularization": 2.0,
        "early_stopping": True,
        "random_state": CONFIG["SEED"] + fold,
    }
    m_hgb = HistGradientBoostingClassifier(**hgb_params) if CONFIG["TASK"] != "regression" else HistGradientBoostingRegressor(**hgb_params)
    m_hgb.fit(X_tr, y_tr)
    p_va_hgb = m_hgb.predict_proba(X_va)[:, 1] if CONFIG["TASK"] == "binary" else m_hgb.predict(X_va)
    p_te_hgb = m_hgb.predict_proba(X_te)[:, 1] if CONFIG["TASK"] == "binary" else m_hgb.predict(X_te)
    oof_preds["hgb"][val_idx] = p_va_hgb
    test_preds["hgb"] += p_te_hgb / n_folds_actual
    fold_scores["hgb"].append(calculate_metric(y_va, p_va_hgb, CONFIG["METRIC"], CONFIG["TASK"]))

    # 5. L2-Regularized Linear Model
    m_lin = LogisticRegression(C=0.1, max_iter=500, random_state=CONFIG["SEED"] + fold) if CONFIG["TASK"] != "regression" else Ridge(alpha=10.0, random_state=CONFIG["SEED"] + fold)
    m_lin.fit(X_tr_lin, y_tr)
    p_va_lin = m_lin.predict_proba(X_va_lin)[:, 1] if CONFIG["TASK"] == "binary" else m_lin.predict(X_va_lin)
    p_te_lin = m_lin.predict_proba(X_te_lin)[:, 1] if CONFIG["TASK"] == "binary" else m_lin.predict(X_te_lin)
    oof_preds["linear"][val_idx] = p_va_lin
    test_preds["linear"] += p_te_lin / n_folds_actual
    fold_scores["linear"].append(calculate_metric(y_va, p_va_lin, CONFIG["METRIC"], CONFIG["TASK"]))

print(f"\n[DONE] 5 Model Families trained across {n_folds_actual} folds in {(time.time() - t0):.1f}s")
l1_oof_scores = {}
for m in model_names:
    sc = calculate_metric(y_raw.values, oof_preds[m], CONFIG["METRIC"], CONFIG["TASK"])
    l1_oof_scores[m] = sc
    print(f"  Level-1 {m.upper():<7} -> Full OOF: {sc:.5f} | Fold Mean: {np.mean(fold_scores[m]):.5f} +/- {np.std(fold_scores[m]):.5f}")


[DONE] 5 Model Families trained across 5 folds in 40.5s
  Level-1 LGB     -> Full OOF: 0.95629 | Fold Mean: 0.95674 +/- 0.00722
  Level-1 CAT     -> Full OOF: 0.95981 | Fold Mean: 0.95988 +/- 0.00910
  Level-1 XGB     -> Full OOF: 0.95464 | Fold Mean: 0.95496 +/- 0.00864
  Level-1 HGB     -> Full OOF: 0.95204 | Fold Mean: 0.95292 +/- 0.00504
  Level-1 LINEAR  -> Full OOF: 0.86062 | Fold Mean: 0.86148 +/- 0.01020


## 8. Komparasi 3 Metode Ensemble Bebas Kebocoran (Plot 03)
Kita mengadu 3 metode *ensembling* standar Grandmaster secara *strictly Out-Of-Fold*:
1. **Greedy Hill Climbing**: Mengoptimalkan bobot penggabungan ($w_i \ge 0, \sum w_i = 1$) secara langsung pada metrik evaluasi kompetisi (`CONFIG["METRIC"]`). Dijamin menghasilkan skor OOF $\ge$ model tunggal terbaik.
2. **Nested Out-Of-Fold Level-2 Stacking**: Melatih model meta (`LogisticRegression` / `Ridge` ter-regularisasi L2) menggunakan `cross_val_predict` pada matriks `S_train`, sehingga evaluasi OOF Level-2 100% bebas *in-sample overfitting*.
3. **Rank-Averaged Trees**: Mengubah prediksi menjadi persentil peringkat untuk menghilangkan distorsi kalibrasi probabilitas.


In [1]:
def greedy_hill_climbing(oof_dict: dict[str, np.ndarray], test_dict: dict[str, np.ndarray], y_true: np.ndarray, metric_name: str, task: str) -> tuple[np.ndarray, np.ndarray, dict[str, float]]:
    """
    Optimasi bobot ensemble secara Greedy Hill Climbing langsung pada metrik kompetisi
    dengan kendala bobot non-negatif (w >= 0, sum(w) = 1).
    """
    higher_better = is_higher_better(metric_name)
    names = list(oof_dict.keys())
    best_name = max(names, key=lambda k: calculate_metric(y_true, oof_dict[k], metric_name, task)) if higher_better else min(names, key=lambda k: calculate_metric(y_true, oof_dict[k], metric_name, task))

    weights = {k: (1.0 if k == best_name else 0.0) for k in names}
    current_oof = oof_dict[best_name].copy()
    current_test = test_dict[best_name].copy()
    best_score = calculate_metric(y_true, current_oof, metric_name, task)

    for _ in range(15):
        improved = False
        best_cand, best_w = None, 0.0
        for name in names:
            for w in np.linspace(0.01, 0.45, 45):
                cand_oof = (1.0 - w) * current_oof + w * oof_dict[name]
                sc = calculate_metric(y_true, cand_oof, metric_name, task)
                if (higher_better and sc > best_score + 1e-6) or (not higher_better and sc < best_score - 1e-6):
                    best_score = sc
                    best_cand = name
                    best_w = float(w)
                    improved = True
        if improved and best_cand is not None:
            current_oof = (1.0 - best_w) * current_oof + best_w * oof_dict[best_cand]
            current_test = (1.0 - best_w) * current_test + best_w * test_dict[best_cand]
            for k in weights:
                weights[k] *= (1.0 - best_w)
            weights[best_cand] += best_w
        else:
            break

    return current_oof, current_test, weights


# Strategy 1: Greedy Hill Climbing
oof_hc, test_hc, hc_weights = greedy_hill_climbing(oof_preds, test_preds, y_raw.values, CONFIG["METRIC"], CONFIG["TASK"])
score_hc = calculate_metric(y_raw.values, oof_hc, CONFIG["METRIC"], CONFIG["TASK"])

# Strategy 2: Nested Out-Of-Fold Level-2 Stacking (Zero Meta-Leakage)
S_train = np.column_stack([oof_preds[m] for m in model_names])
S_test = np.column_stack([test_preds[m] for m in model_names])

if CONFIG["TASK"] in ["binary", "multiclass"]:
    meta_model = LogisticRegression(C=0.5, max_iter=500, random_state=CONFIG["SEED"])
    oof_stack = cross_val_predict(
        meta_model, S_train, y_raw.values,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG["SEED"]),
        method="predict_proba"
    )[:, 1]
    meta_model.fit(S_train, y_raw.values)
    test_stack = meta_model.predict_proba(S_test)[:, 1]
else:
    meta_model = Ridge(alpha=5.0, random_state=CONFIG["SEED"])
    oof_stack = cross_val_predict(
        meta_model, S_train, y_raw.values,
        cv=KFold(n_splits=5, shuffle=True, random_state=CONFIG["SEED"])
    )
    meta_model.fit(S_train, y_raw.values)
    test_stack = meta_model.predict(S_test)

score_stack = calculate_metric(y_raw.values, oof_stack, CONFIG["METRIC"], CONFIG["TASK"])

# Strategy 3: Rank Averaging (Top 4 Tree Models)
tree_models = ["lgb", "cat", "xgb", "hgb"]
oof_rank = rank_average([oof_preds[m] for m in tree_models])
test_rank = rank_average([test_preds[m] for m in tree_models])
score_rank = calculate_metric(y_raw.values, oof_rank, CONFIG["METRIC"], CONFIG["TASK"])

print("\n--- ENSEMBLE COMPARISON (100% OUT-OF-FOLD) ---")
for m, w in hc_weights.items():
    print(f"  Hill-Climbing Weight [{m.upper():<6}]: {w:.4f}")
print(f"  1. Hill-Climbing Ensemble OOF : {score_hc:.5f}")
print(f"  2. Nested L2 Stacking OOF     : {score_stack:.5f}")
print(f"  3. Rank-Averaged Trees OOF    : {score_rank:.5f}")

candidates = {
    "Hill-Climbing": (score_hc, oof_hc, test_hc),
    "Nested-Stacking": (score_stack, oof_stack, test_stack),
    "Rank-Average": (score_rank, oof_rank, test_rank),
}
best_ens_name = max(candidates.keys(), key=lambda k: candidates[k][0]) if is_higher_better(CONFIG["METRIC"]) else min(candidates.keys(), key=lambda k: candidates[k][0])
best_ens_score, best_oof, best_test = candidates[best_ens_name]
print(f"[CHAMPION ENSEMBLE] {best_ens_name} selected with OOF {CONFIG['METRIC'].upper()} = {best_ens_score:.5f}")

best_th, best_f1 = optimize_threshold(y_raw.values, best_oof, metric_name="f1")
print(f"Optimal Binary Classification Threshold: {best_th:.2f} (OOF F1-Score: {best_f1:.5f})")

# Plot 03: Model Zoo & Ensemble Comparison + Weights
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
all_labels = [m.upper() for m in model_names] + ["RANK AVG", "NESTED STACK", "HILL CLIMB"]
all_scores = [l1_oof_scores[m] for m in model_names] + [score_rank, score_stack, score_hc]
colors = ["#3498db", "#2ecc71", "#e67e22", "#1abc9c", "#95a5a6", "#f39c12", "#9b59b6", "#e74c3c"]

bars = axes[0].barh(all_labels, all_scores, color=colors)
axes[0].set_xlim(min(all_scores) * 0.96, max(all_scores) * 1.015)
axes[0].set_title(f"OOF Progression: Base Models vs Ensembles ({CONFIG['METRIC'].upper()})", fontsize=11.5, weight="bold")
axes[0].set_xlabel(f"Out-Of-Fold {CONFIG['METRIC'].upper()}")
for bar in bars:
    w_val = bar.get_width()
    axes[0].text(w_val, bar.get_y() + bar.get_height() / 2, f" {w_val:.5f}", va="center", ha="left", fontsize=9, fontweight="bold")

w_labels = [m.upper() for m in hc_weights.keys()]
w_vals = [hc_weights[m] for m in hc_weights.keys()]
sns.barplot(x=w_vals, y=w_labels, ax=axes[1], palette="viridis")
axes[1].set_xlim(0, 1.05)
axes[1].set_title("Optimal Hill-Climbing Blend Weights", fontsize=11.5, weight="bold")
axes[1].set_xlabel("Assigned Ensemble Weight")
for idx, v in enumerate(w_vals):
    axes[1].text(v + 0.02, idx, f"{v*100:.1f}%", va="center", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["EXPORT_DIR"], "03_model_zoo_and_ensemble_comparison.png"), dpi=300)
plt.show()


--- ENSEMBLE COMPARISON (100% OUT-OF-FOLD) ---
  Hill-Climbing Weight [LGB   ]: 0.2200
  Hill-Climbing Weight [CAT   ]: 0.7800
  Hill-Climbing Weight [XGB   ]: 0.0000
  Hill-Climbing Weight [HGB   ]: 0.0000
  Hill-Climbing Weight [LINEAR]: 0.0000
  1. Hill-Climbing Ensemble OOF : 0.96007
  2. Nested L2 Stacking OOF     : 0.95974
  3. Rank-Averaged Trees OOF    : 0.95744
[CHAMPION ENSEMBLE] Hill-Climbing selected with OOF ROC_AUC = 0.96007
Optimal Binary Classification Threshold: 0.47 (OOF F1-Score: 0.89542)


## 9. Diagnostik Model, Kalibrasi, Interpretabilitas & Audit Distribusi Prediksi (Plot 04, 05, 06)
Untuk memenangkan **30% Presentasi**, kita menghasilkan 3 visualisasi analisis mendalam:
* `04_diagnostics_cm_and_calibration.png`: *Confusion Matrix* pada ambang batas optimal beserta kurva keandalan probabilitas (*Reliability / Calibration Curve* dan skor Brier).
* `05_feature_importance_and_shap.png`: Konsensus 15 fitur terpenting lintas seluruh *fold* CV untuk rekomendasi bisnis.
* `06_oof_vs_test_prediction_distribution.png`: Audit visual kesesuaian bentuk distribusi prediksi OOF Train vs Test sebelum *submit*.


In [ ]:
# Plot 04: Confusion Matrix & Probability Calibration Curve
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
if CONFIG["TASK"] == "binary":
    y_pred_bin = (best_oof >= best_th).astype(int)
    cm = confusion_matrix(y_raw.values, y_pred_bin)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0], annot_kws={"size": 13, "weight": "bold"})
    axes[0].set_title(f"OOF Confusion Matrix (Threshold = {best_th:.2f})", fontsize=12, weight="bold")
    axes[0].set_xlabel("Predicted Class")
    axes[0].set_ylabel("True Class")

    prob_true, prob_pred = calibration_curve(y_raw.values, np.clip(best_oof, 0, 1), n_bins=10)
    brier = brier_score_loss(y_raw.values, np.clip(best_oof, 0, 1))
    axes[1].plot(prob_pred, prob_true, marker="o", lw=2, color="#2980b9", label=f"Ensemble (Brier = {brier:.4f})")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly Calibrated")
    axes[1].set_title("Probability Calibration Curve (Reliability)", fontsize=12, weight="bold")
    axes[1].set_xlabel("Mean Predicted Probability")
    axes[1].set_ylabel("Fraction of Positives")
    axes[1].legend(loc="lower right")
else:
    axes[0].scatter(y_raw.values, best_oof, alpha=0.4, color="#2980b9")
    axes[0].plot([y_raw.min(), y_raw.max()], [y_raw.min(), y_raw.max()], "r--", lw=2)
    axes[0].set_title("Actual vs Predicted (OOF)", fontsize=12, weight="bold")
    residuals = y_raw.values - best_oof
    sns.histplot(residuals, kde=True, ax=axes[1], color="#e74c3c")
    axes[1].set_title("OOF Residual Error Distribution", fontsize=12, weight="bold")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["EXPORT_DIR"], "04_diagnostics_cm_and_calibration.png"), dpi=300)
plt.show()

# Plot 05: Consensus Feature Importance across Fold Models
mean_lgb_imp = np.mean([m.feature_importances_ for m in fitted_lgb_models], axis=0)
imp_series = pd.Series(mean_lgb_imp, index=last_val_X.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 5.2))
sns.barplot(x=imp_series.values, y=imp_series.index, palette="viridis")
plt.title("Top 15 Consensus Feature Importances (Across All CV Folds)", fontsize=12.5, weight="bold")
plt.xlabel("Mean Split Importance Across Folds")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["EXPORT_DIR"], "05_feature_importance_and_shap.png"), dpi=300)
plt.show()

# Plot 06: Prediction Distribution Alignment (OOF vs Test Sanity Check)
plt.figure(figsize=(8, 4.2))
sns.kdeplot(best_oof, fill=True, color="#2980b9", alpha=0.35, lw=2, label="Train OOF Predictions")
sns.kdeplot(best_test, fill=True, color="#e74c3c", alpha=0.35, lw=2, label="Test Set Predictions")
plt.title("Sanity Check: Train OOF vs Test Prediction Distribution", fontsize=12, weight="bold")
plt.xlabel("Predicted Score / Probability")
plt.ylabel("Density")
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["EXPORT_DIR"], "06_oof_vs_test_prediction_distribution.png"), dpi=300)
plt.show()


## 10. Ekspor 3 File Submission & Verifikasi Integritas Ketat
Menghasilkan 3 opsi file *submission* dan melakukan pengecekan otomatis (`assert`) terhadap jumlah baris, ketiadaan nilai `NaN`, serta kevalidan angka hingga mengarsipkan 6 gambar presentasi ke dalam `presentation_assets_ultimate.zip`.


In [1]:
final_test_hc = (test_hc >= best_th).astype(int) if CONFIG["METRIC"] in ["f1", "f1_macro", "accuracy"] else test_hc
final_test_stack = (test_stack >= best_th).astype(int) if CONFIG["METRIC"] in ["f1", "f1_macro", "accuracy"] else test_stack

sub_1 = pd.DataFrame({CONFIG["ID_COL"]: df_test_clean[CONFIG["ID_COL"]], CONFIG["TARGET_COL"]: final_test_hc})
sub_2 = pd.DataFrame({CONFIG["ID_COL"]: df_test_clean[CONFIG["ID_COL"]], CONFIG["TARGET_COL"]: final_test_stack})
sub_3 = pd.DataFrame({CONFIG["ID_COL"]: df_test_clean[CONFIG["ID_COL"]], CONFIG["TARGET_COL"]: test_rank})

sub_1.to_csv("sub_1_best_hill_climbing.csv", index=False)
sub_2.to_csv("sub_2_nested_stacking.csv", index=False)
sub_3.to_csv("sub_3_rank_average.csv", index=False)

for fname, s_df in [("sub_1_best_hill_climbing.csv", sub_1), ("sub_2_nested_stacking.csv", sub_2), ("sub_3_rank_average.csv", sub_3)]:
    assert len(s_df) == len(df_test), f"[ERROR] Row count mismatch in {fname}"
    assert s_df[CONFIG["TARGET_COL"]].isnull().sum() == 0, f"[ERROR] NaN detected in {fname}"
    assert np.all(np.isfinite(s_df[CONFIG["TARGET_COL"]].values)), f"[ERROR] Infinite values in {fname}"

shutil.make_archive("presentation_assets_ultimate", "zip", CONFIG["EXPORT_DIR"])
print("[VERIFIED] All 3 submission files passed strict integrity checks (0 NaNs, exact row count).")
print("[VERIFIED] 6 presentation figures (300 DPI) archived to: presentation_assets_ultimate.zip")


[VERIFIED] All 3 submission files passed strict integrity checks (0 NaNs, exact row count).
[VERIFIED] 6 presentation figures (300 DPI) archived to: presentation_assets_ultimate.zip


### Q&A
* **Mengapa fitur frekuensi, grup agregasi, Target Encoding, PCA, dan K-Means dimasukkan ke dalam `InFoldFeaturePipeline` dan bukan dihitung di awal pada seluruh tabel?**
  Menghitung rata-rata target, rata-rata grup, atau koordinat klaster pada keseluruhan data *train* sebelum pemisahan *fold* menyebabkan informasi dari data validasi merembes ke data latih (*data leakage*). Dengan membungkus seluruh transformasi berbasis distribusi di dalam `InFoldFeaturePipeline`, setiap *fold* validasi diperlakukan murni seperti data *test* yang belum pernah terlihat.
* **Mengapa kita membandingkan Greedy Hill Climbing dan Nested L2 Stacking?**
  Model *stacking* standar sering mengalami penurunan performa jika dievaluasi secara *in-sample* atau jika fungsi *loss* meta-learner (misalnya *log-loss*) tidak sejajar dengan metrik kompetisi (misalnya *ROC-AUC* atau *F1*). *Greedy Hill Climbing* mengoptimalkan metrik kompetisi secara langsung dengan kendala bobot non-negatif ($w_i \ge 0, \sum w_i = 1$), sehingga dijamin tidak pernah lebih buruk daripada model tunggal terbaik.

### Data Analysis Key Findings
* **Validasi Konsistensi Distribusi Data**: Pengujian *Adversarial Validation* antara data *train* (2.500 baris) dan *test* (700 baris) menghasilkan skor ROC-AUC sebesar **0.4983** (mendekati nilai acak ideal 0.5000), membuktikan ketiadaan *covariate shift* antara kedua partisi data.
* **Performa 5 Keluarga Model Level-1 (Out-Of-Fold)**: Dari 5 arsitektur yang dilatih pada 5-Fold Stratified CV, **CatBoost** mencatat skor OOF tertinggi sebesar **0.95981** (rata-rata fold $0.95988 \pm 0.00910$), diikuti oleh **LightGBM** (**0.95629**), **XGBoost** (**0.95464**), **HistGradientBoosting** (**0.95204**), dan **Regularized Linear** (**0.86062**).
* **Peningkatan Akurasi melalui Hill-Climbing Ensemble**: Penggabungan prediksi OOF via *Greedy Hill Climbing* (bobot optimal **78.0% CatBoost + 22.0% LightGBM**) meningkatkan skor ROC-AUC menjadi **0.96007**, mengungguli model tunggal terbaik maupun *Nested L2 Stacking* (**0.95974**). Optimasi ambang batas keputusan (*threshold tuning*) menemukan titik potong optimal pada **0.47** dengan skor OOF F1 sebesar **0.89542**.

### Insights or Next Steps
* **Fokus Kustomisasi Saat Lomba Dimulai**: Tim Anda cukup berfokus menambahkan fitur rasio bisnis dan pembersihan anomali spesifik domain pada fungsi `user_custom_preprocessing` dan `user_custom_feature_engineering` di **Sel 4**, karena seluruh proses *encoding* berisiko tinggi sudah diamankan di dalam *fold* CV.
* **Strategi Pemilihan 2 File Final untuk Private Leaderboard**: Gunakan `sub_1_best_hill_climbing.csv` sebagai *submission* utama (skor CV lokal tertinggi) dan `sub_3_rank_average.csv` sebagai *submission* cadangan karena rata-rata peringkat (*rank averaging*) kebal terhadap pergeseran kalibrasi probabilitas di *Private Leaderboard*.
